# IoT Challenge #3 - Exercise (EQ1, EQ2)

This notebook solves the LoRaWAN exercise with a simple overlap/collision model (pure ALOHA-style).

## Given
- Region/frequency: EU868 (868 MHz)
- Bandwidth: 125 kHz
- Nodes: 40
- Per-node traffic: $\lambda = 2$ packets/minute
- Payload size: 20 bytes

Collision model:
- all nodes use the same SF
- uncoordinated transmissions
- a packet is lost if another packet overlaps it in time
- vulnerable window is approximately $2T$ (where $T$ is packet airtime).

In [ ]:
import math
from dataclasses import dataclass

# LoRa airtime formula consistent with the TTN-style calculator inputs
def lora_airtime_seconds(sf, pl_bytes, bw_hz=125_000, cr=1, preamble=8, crc=1, ih=0, de=None):
    if de is None:
        de = 1 if (bw_hz == 125_000 and sf >= 11) else 0

    t_sym = (2 ** sf) / bw_hz
    t_preamble = (preamble + 4.25) * t_sym

    num = 8 * pl_bytes - 4 * sf + 28 + 16 * crc - 20 * ih
    den = 4 * (sf - 2 * de)
    payload_symb = 8 + max(math.ceil(num / den) * (cr + 4), 0)
    t_payload = payload_symb * t_sym

    return t_preamble + t_payload

def success_prob(sf, payload_bytes=20, nodes=40, lam_per_node_per_min=2):
    T = lora_airtime_seconds(sf, payload_bytes)

    # Interferers are other nodes only
    lambda_others = (nodes - 1) * lam_per_node_per_min / 60.0  # packets/s

    # P(success) = exp(-lambda_others * 2T)
    p = math.exp(-lambda_others * 2.0 * T)
    return T, p

rows = []
for sf in range(7, 13):
    T, p = success_prob(sf)
    rows.append((sf, T * 1000, p))

rows

In [ ]:
print(f"{'SF':<4}{'Airtime (ms)':<16}{'Success prob':<14}")
for sf, t_ms, p in rows:
    print(f"{sf:<4}{t_ms:<16.3f}{p:<14.4f}")

valid = [sf for sf, t_ms, p in rows if p >= 0.75]
largest_sf = max(valid) if valid else None
print('\nLargest SF with success >= 75%:', largest_sf)

## EQ1 Answer
From the table, the largest SF that still keeps packet success probability at or above 75% is **SF8**.

Reason: SF9 already pushes airtime too high, increasing overlap probability too much.

## EQ2 Answer
Most appropriate action: **2. Move the nodes closer to the gateway**.

Why:
- The measured non-uniform performance across nodes suggests link-quality differences (path loss, obstacles, fading), not just average traffic load.
- Moving nodes closer improves SNR/RSSI and reduces capture/decoding failures for weak nodes.
- Decreasing node count may reduce collisions but does not directly fix weak-link nodes.
- Increasing SF increases airtime, which typically increases collision probability and network load.

In [ ]:
# quick consistency check for SF8 and SF9
for sf in [8, 9]:
    T, p = success_prob(sf)
    print(f"SF{sf}: airtime={T*1000:.3f} ms, success={p:.4f}")